### Dataset Architecture & Feature Specification

A **hybrid dataset** integrating **119,404 real-world physical records** from Kaggle's *Clothes Size Prediction* dataset, complemented by synthetic garment parameters and target sizing logic.

#### 1. Data Generation & Partitioning
* **Data Origin:** Real human physical dimensions (`height`, `weight`) extracted directly from Kaggle's `final_test.csv` ($N = 119,404$ records).
* **Complemented Features:** Synthetic garment specifications (`garment_chest_cm`, `fabric_stretch_pct`, `product_type_id`) generated to represent retail inventory.
* **Sampling & Partitioning:** To ensure fast optimization during Grid Search, a stratified subset of 15,000 records was sampled and split:
  * **Training Set:** 12,000 samples (80%)
  * **Test Set:** 3,000 samples (20%)
* **Target Classes:** Fit outcomes derived from chest-to-garment physical dimension differentials:
  * `0`: **Too Small** (Garment chest smaller than estimated body chest)
  * `1`: **Good Fit** (Garment chest matches body chest within $\pm 4.0\text{ cm}$)
  * `2`: **Too Large** (Garment chest larger than estimated body chest)

#### 2. Feature Specifications & Physical Constraints

| Feature Name | Data Type | Data Source / Distribution | Physical Bounds | Description |
| :--- | :--- | :--- | :--- | :--- |
| **`height_cm`** | Continuous | Kaggle real data (`height` column) | 140.0 cm to 220.0 cm | User physical height. |
| **`weight_kg`** | Continuous | Kaggle real data (`weight` column) | 40.0 kg to 150.0 kg | User physical weight. |
| **`imc_index`** | Continuous | Calculated feature | Dynamic (10.0 to 50.0+) | Derived Body Mass Index (BMI). |
| **`garment_chest_cm`** | Continuous | Synthesized ($\mu=95.0, \sigma=15.0$) | 60.0 cm to 140.0 cm | Flat chest width of garment. |
| **`fabric_stretch_pct`** | Continuous | Synthesized ($\mu=5.0, \sigma=5.0$) | 0.0% to 20.0% | Elasticity factor of material. |
| **`product_type_id`** | Categorical | Discrete integers (`0`, `1`, `2`) | `0`, `1`, `2` | Garment category (0: Tops, 1: Bottoms, 2: Jackets). |

#### 3. Data Transformations & Feature Engineering
* **Derived BMI (`imc_index`):** Computed dynamically using body dimensions:
  $$\text{imc\_index} = \frac{\text{weight\_kg}}{\left(\frac{\text{height\_cm}}{100}\right)^2}$$

* **Heuristic Target Labeling:** Target `fit_class` labels are created using estimated chest clearance:
  $$\text{estimated\_body\_chest} = 0.45 \times \text{weight\_kg} + 0.35 \times \text{height\_cm}$$
  $$\text{chest\_delta} = (\text{garment\_chest\_cm} + 0.5 \times \text{fabric\_stretch\_pct}) - \text{estimated\_body\_chest}$$

* **One-Hot Encoding:** `product_type_id` is encoded via `pd.get_dummies()`, expanding the schema to **8 feature columns** stored in `model_features.joblib`:
  1. `height_cm`
  2. `weight_kg`
  3. `imc_index`
  4. `garment_chest_cm`
  5. `fabric_stretch_pct`
  6. `product_type_id_0`
  7. `product_type_id_1`
  8. `product_type_id_2`

In [16]:
!pip install kagglehub -q

import kagglehub
import os
import shutil

# Download dataset from Kaggle
path = kagglehub.dataset_download("tourist55/clothessizeprediction")

# Move the downloaded file to current working directory as final_test.csv
for file in os.listdir(path):
    if file.endswith(".csv"):
        shutil.copy(os.path.join(path, file), "final_test.csv")
        print(f"Loaded {file} as final_test.csv")

Using Colab cache for faster access to the 'clothessizeprediction' dataset.
Loaded final_test.csv as final_test.csv


In [17]:
import pandas as pd
import numpy as np

# ==========================================
# TASK 1: HYBRID DATASET (Kaggle Real Data + Complemented Features)
# ==========================================

# 1. Load Kaggle dataset (Ensure 'final_test.csv' or 'cloth_size.csv' is uploaded)
kaggle_raw = pd.read_csv("final_test.csv").dropna(subset=["height", "weight"])

np.random.seed(42)
n_samples = len(kaggle_raw)

# 2. Extract real physical human dimensions
height_cm = np.clip(kaggle_raw["height"].values, 140.0, 220.0)
weight_kg = np.clip(kaggle_raw["weight"].values, 40.0, 150.0)

# 3. Calculate derived BMI (imc_index)
imc_index = weight_kg / ((height_cm / 100.0) ** 2)

# 4. Complement missing garment & apparel features
garment_chest_cm = np.clip(np.random.normal(95.0, 15.0, size=n_samples), 60.0, 140.0)
fabric_stretch_pct = np.clip(np.random.normal(5.0, 5.0, size=n_samples), 0.0, 20.0)
product_type_id = np.random.choice([0, 1, 2], size=n_samples)

# 5. Generate realistic fit_class target based on body vs garment chest differential
estimated_body_chest = 0.45 * weight_kg + 0.35 * height_cm
chest_delta = (garment_chest_cm + (fabric_stretch_pct * 0.5)) - estimated_body_chest

# Target labels: 0 = Too Small, 1 = Good Fit, 2 = Too Large
fit_class = np.where(chest_delta < -4.0, 0, np.where(chest_delta > 4.0, 2, 1))

# 6. Define schema matching downstream ML pipeline
feature_names = [
    'height_cm', 'weight_kg', 'imc_index',
    'garment_chest_cm', 'fabric_stretch_pct', 'product_type_id'
]

fitcheck_df = pd.DataFrame({
    'height_cm': height_cm,
    'weight_kg': weight_kg,
    'imc_index': imc_index,
    'garment_chest_cm': garment_chest_cm,
    'fabric_stretch_pct': fabric_stretch_pct,
    'product_type_id': product_type_id,
    'fit_class': fit_class
})

print(f"Hybrid dataset created with {len(fitcheck_df)} real Kaggle records!")

Hybrid dataset created with 119404 real Kaggle records!


In [18]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. One-Hot Encode product_type_id from the hybrid fitcheck_df
X_encoded = pd.get_dummies(
    fitcheck_df[['height_cm', 'weight_kg', 'imc_index', 'garment_chest_cm', 'fabric_stretch_pct', 'product_type_id']],
    columns=['product_type_id'],
    drop_first=False
)
y = fitcheck_df['fit_class']

# 2. Downsample for fast Grid Search execution (optional, use 15,000 rows for speed)
# With 119,404 rows, grid search on the full dataset can take 15+ minutes
X_sample, _, y_sample, _ = train_test_split(X_encoded, y, train_size=15000, random_state=42, stratify=y)

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

# 4. Define and run Grid Search
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8]
}

print("Running Grid Search on hybrid dataset...")
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# 5. Extract best model
best_gb_model = grid_search.best_estimator_
y_pred = best_gb_model.predict(X_test)

print(f"Optimized Accuracy: {accuracy_score(y_test, y_pred):.3f}")

# 6. Save model and feature artifacts
joblib.dump(best_gb_model, 'fitcheck_gb_model.joblib')
model_features = list(X_encoded.columns)
joblib.dump(model_features, 'model_features.joblib')

print("Model and feature specs saved successfully!")

Running Grid Search on hybrid dataset...
Optimized Accuracy: 0.972
Model and feature specs saved successfully!


**1. Save the Model Artifact and Feature Columns**\
To deploy this as a microservice, the trained model and the feature column structure saved to ensure consistent one-hot encoding during live inference:

In [19]:
# Save the trained model artifact
joblib.dump(best_gb_model, 'fitcheck_gb_model.joblib')

# Save expected feature column names for live inference matching
model_features = list(X_encoded.columns)
joblib.dump(model_features, 'model_features.joblib')

print("Model and feature specs saved successfully!")

Model and feature specs saved successfully!


**2. Creating an Inference Pipeline Helper** \
A lightweight prediction function that takes a raw user profile, applies the necessary transformations (derived BMI + One-Hot Encoding), and outputs a prediction that can be passed directly into our Gemini/MLflow prompt:

In [20]:
def predict_fit(user_data, model, feature_columns):
    """
    user_data dict structure:
    {
        'height_cm': 175.0,
        'weight_kg': 72.0,
        'garment_chest_cm': 98.0,
        'fabric_stretch_pct': 3.0,
        'product_type_id': 0  # 0: Tops, 1: Bottoms, 2: Jackets
    }
    """
    df = pd.DataFrame([user_data])

    # Calculate derived BMI
    df['imc_index'] = df['weight_kg'] / ((df['height_cm'] / 100.0) ** 2)

    # One-hot encode product type
    df_encoded = pd.get_dummies(df, columns=['product_type_id'])

    # Reindex to match training schema (fills missing dummy columns with 0)
    df_encoded = df_encoded.reindex(columns=feature_columns, fill_value=0)

    # Class mapping
    class_map = {0: "Too Small", 1: "Good Fit", 2: "Too Large"}
    prediction_id = model.predict(df_encoded)[0]
    probabilities = model.predict_proba(df_encoded)[0]

    return {
        "prediction_class": class_map[prediction_id],
        "confidence": float(probabilities[prediction_id]),
        "raw_class_id": int(prediction_id)
    }

# Quick Test Example:
sample_user = {
    'height_cm': 180.0,
    'weight_kg': 85.0,
    'garment_chest_cm': 92.0,  # Chest size might be small for an 85kg person
    'fabric_stretch_pct': 2.0,
    'product_type_id': 0
}

result = predict_fit(sample_user, best_gb_model, X_encoded.columns)
print("Sample Live Prediction:", result)

Sample Live Prediction: {'prediction_class': 'Too Small', 'confidence': 0.8542776212688131, 'raw_class_id': 0}


**Model Serialization & MLflow Tracking**\
Registering the model artifact and track the run using MLflow

In [21]:
!pip install mlflow -q

import mlflow
import mlflow.sklearn
import joblib
from sklearn.metrics import accuracy_score

# Ensure y_pred exists from best_gb_model
y_pred = best_gb_model.predict(X_test)

# Set MLflow experiment name
mlflow.set_experiment("FitCheck_Sizing_Model")

with mlflow.start_run(run_name="Gradient_Boosting_Hybrid"):
    # 1. Log Hyperparameters
    mlflow.log_params(best_gb_model.get_params())

    # 2. Log Performance Metrics
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))

    # 3. Log Model Artifact using cloudpickle serialization
    mlflow.sklearn.log_model(
        sk_model=best_gb_model,
        name="model",
        serialization_format="cloudpickle",
        registered_model_name="FitCheck_GBClassifier"
    )

    # 4. Save local backup artifacts
    joblib.dump(best_gb_model, "fitcheck_gb_model.joblib")
    joblib.dump(list(X_encoded.columns), "model_features.joblib")

print("Model tracked and registered in MLflow!")

2026/09/27 13:59:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model tracked and registered in MLflow!


Registered model 'FitCheck_GBClassifier' already exists. Creating a new version of this model...
Created version '2' of model 'FitCheck_GBClassifier'.


In [24]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd

app = FastAPI(title="FitCheck Microservice", version="1.0.0")

# Load trained model and features schema
try:
    model = joblib.load("fitcheck_gb_model.joblib")
    model_features = joblib.load("model_features.joblib")
except Exception:
    model = None
    model_features = []

class PredictionInput(BaseModel):
    height_cm: float
    weight_kg: float
    garment_chest_cm: float
    fabric_stretch_pct: float
    product_type_id: int

@app.get("/health")
def health_check():
    return {
        "status": "online",
        "model_loaded": model is not None,
        "expected_features_count": len(model_features)
    }

@app.post("/predict")
def predict(data: PredictionInput):
    if model is None:
        raise HTTPException(status_code=500, detail="Model artifacts not loaded")

    # 1. Calculate dynamic BMI
    imc_index = data.weight_kg / ((data.height_cm / 100.0) ** 2)

    # 2. Map one-hot encoding structure
    input_dict = {
        "height_cm": data.height_cm,
        "weight_kg": data.weight_kg,
        "imc_index": imc_index,
        "garment_chest_cm": data.garment_chest_cm,
        "fabric_stretch_pct": data.fabric_stretch_pct,
        "product_type_id_0": 1 if data.product_type_id == 0 else 0,
        "product_type_id_1": 1 if data.product_type_id == 1 else 0,
        "product_type_id_2": 1 if data.product_type_id == 2 else 0,
    }

    # 3. Align DataFrame schema with training features
    df_input = pd.DataFrame([input_dict]).reindex(columns=model_features, fill_value=0)

    # 4. Predict class & probability score
    prediction_id = int(model.predict(df_input)[0])
    probabilities = model.predict_proba(df_input)[0]
    confidence = float(probabilities[prediction_id])

    # 5. Sizing class label mapping
    class_map = {0: "Too Small", 1: "Good Fit", 2: "Too Large"}

    return {
        "prediction_class": class_map.get(prediction_id, "Unknown"),
        "confidence": round(confidence, 4),
        "raw_class_id": prediction_id,
        "imc_index": round(imc_index, 2)
    }

Overwriting app.py


**Building and testing the FastAPI Microservice**

1. File Setup (app.py)
2. Running the Microservice Server

In [25]:
!ls -lh app.py requirements.txt Dockerfile fitcheck_gb_model.joblib model_features.joblib

ls: cannot access 'requirements.txt': No such file or directory
ls: cannot access 'Dockerfile': No such file or directory
-rw-r--r-- 1 root root 2.1K Sep 27 14:01 app.py
-rw-r--r-- 1 root root 767K Sep 27 13:59 fitcheck_gb_model.joblib
-rw-r--r-- 1 root root  152 Sep 27 13:59 model_features.joblib


In [27]:
import sys
from fastapi.testclient import TestClient

# 1. Force Python to drop the cached version of app from memory
if "app" in sys.modules:
    del sys.modules["app"]

# 2. Re-import the fresh app.py file written to disk
from app import app

# 3. Re-initialize the TestClient with the updated app instance
client = TestClient(app)

# 4. Test Health Check Endpoint
health_res = client.get("/health")
print("Health Check:", health_res.status_code, health_res.json())

# 5. Test Prediction Endpoint
payload = {
    "height_cm": 182.0,
    "weight_kg": 88.0,
    "garment_chest_cm": 90.0,
    "fabric_stretch_pct": 2.0,
    "product_type_id": 0  # Tops
}

pred_res = client.post("/predict", json=payload)
print("Prediction Response Status:", pred_res.status_code)
print("Prediction JSON:", pred_res.json())

Health Check: 200 {'status': 'online', 'model_loaded': True, 'expected_features_count': 8}
Prediction Response Status: 200
Prediction JSON: {'prediction_class': 'Too Small', 'confidence': 0.9988, 'raw_class_id': 0, 'imc_index': 26.57}
